# Distributed PyTorch & Transformer Implementation

This notebook contains a from-scratch implementation of a Transformer language model (`GPTTransformer`), focusing on the core mechanics without the complexity of production models (e.g. no KV caching, dropout, or mixed precision). 

Additionally, it benchmarks the training process on a single GPU versus a multi-GPU setup using PyTorch's Distributed Data Parallel (`DDP`).

### Authorship Markers
To maintain transparency on what was hand-written versus what was assisted by AI, the sections are annotated with the following markers:
- `> ✍️ **My Work**` — The core architecture (Linear, Embedding, MLP, LayerNorm, MultiHeadAttention, GPTTransformer) which I implemented myself to build intuition.
- `> 🤖 **AI-Assisted**` — Boilerplate code such as imports, data loading, positional encoding, and the distributed computing logic (adapted from a YouTube tutorial).

> 🤖 **AI-Assisted**

### 1. Standard PyTorch Imports
Importing the core PyTorch libraries needed for building the neural network components from scratch.

In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import math
import time

> 🤖 **AI-Assisted**

### 2. Distributed Training Imports
Importing PyTorch's multiprocessing and Distributed Data Parallel (DDP) modules. These are essential for scaling the model across multiple GPUs.

In [3]:
#multi gpu imports

import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
import os

> 🤖 **AI-Assisted**

### 3. Data Loading & Tokenization
Downloading the Tiny Shakespeare dataset and creating a custom PyTorch `Dataset` and `DataLoader`. Notice the character-level tokenization (`vocab_size = 65`) to keep things simple.

In [4]:
import urllib.request
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Download Tiny Shakespeare
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, 'shakespeare.txt')

with open('shakespeare.txt', 'r') as f:
    text = f.read()

# 2. Simple Character-Level Tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Dictionaries to map characters to integers and back
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encoder and Decoder functions
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Convert entire dataset to a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)

# 3. Create a Custom PyTorch Dataset
class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
        
    def __len__(self):
        # We subtract seq_len so we don't go out of bounds at the end of the text
        return len(self.data) - self.seq_len
        
    def __getitem__(self, idx):
        # Input sequence (Context Window)
        x = self.data[idx : idx + self.seq_len]
        
        # Target sequence (Shifted right by 1 character to predict the next token)
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        
        return x, y

# 4. Create the PyTorch DataLoader
seq_len = 512    # Your context window
batch_size = 32  # How many sequences to process at once
data = data[0:10000]
dataset = CharDataset(data, seq_len)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# --- Test the DataLoader ---
x_batch, y_batch = next(iter(dataloader))
print("Dataset Loaded Successfully!")
print(f"Vocab size: {vocab_size}")
print(f"x_batch shape: {x_batch.shape}")
print(f"y_batch shape: {y_batch.shape}")

Dataset Loaded Successfully!
Vocab size: 65
x_batch shape: torch.Size([32, 512])
y_batch shape: torch.Size([32, 512])


> ✍️ **My Work**

### 4. Device Selection
Setting the default device to GPU 0 for the single-GPU baseline.

In [5]:
device = 0

> ✍️ **My Work**

### 5. Custom Linear Layer
Implementing a standard linear (dense) layer using raw `nn.Parameter` tensors for weights and biases, instead of using `nn.Linear`.

In [ ]:
class Linear(nn.Module): 
  def __init__(self, d_in= 512, d_out=1024, last = False):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.float
                        ))
        self.b = nn.Parameter(
                        torch.zeros(
                                size = (d_out, ),
                                requires_grad=True, 
                                dtype = torch.float
                        ))

        self.last = last
  def act(self, x): 
    if self.last: 
      return x
    else: 
      return F.relu(x)
     
  def forward(self, x): 
    #x -> (B, d_in) @ ( d_in, d_out)
    return self.act(torch.matmul(x, self.W ) + self.b)

> ✍️ **My Work**

### 6. Custom Embedding Layer
An embedding layer implemented by converting input indices to one-hot vectors and multiplying by a weight matrix.

In [12]:
class GatherEmbedding(nn.Module): 
    def __init__(self, d_in= 10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.float
                        ))
        self.d_in = d_in
        self.d_out = d_out 
    
    def forward(self, x): 
        batch_size, seq_len = x.shape

        # get the values for the rows we want and duplicate them d_out times to get all of the values 
        grabber = x.flatten().unsqueeze(1).expand( -1, self.d_out )
        # gather then go back to 3d 
        return self.W.gather(dim = 0, index = grabber).reshape(batch_size, seq_len, self.d_out)
                


In [13]:
class SparseEmbedding(nn.Module): 
    def __init__(self, d_in= 10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.float
                        ))
        self.d_in = d_in
    
    def forward(self, x): 
        batch_size, seq_len = x.shape

        row_idx = torch.arange(0,batch_size*seq_len)
       
        idx = torch.stack([row_idx.flatten(),x.flatten()])

        vals = torch.ones((batch_size*  seq_len))
         #squash to batch_size*seq_len, self.d_in
        one_hot_sparse = torch.sparse_coo_tensor(idx, vals, size = (batch_size*seq_len,self.d_in ))
        # multiply in 2D
        #(batch* seq_len,v) x (v, d_model) -> (batch * seq_len, d_model)
        O= one_hot_sparse @ self.W
        # reconstruct 
        
        return O.reshape(batch_size, seq_len , -1)


In [14]:
class OneHotEmbedding(nn.Module): 
    def __init__(self, d_in= 10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.float
                        ))
        self.d_in = d_in
    def forward(self, x): 
        #encode in one hot to access embedding vector
        x= F.one_hot(x, num_classes=self.d_in).to(torch.float)
        #(batch, v) x (v, d_model)
        return x.matmul(self.W)

In [15]:
class NativeEmbedding(nn.Module): 
    def __init__(self, d_in=10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
    
    def forward(self, x): 
        return self.W[x]


In [17]:
import torch
import time

def run_four_way_benchmark():
    # Force to CPU to ensure all 4 backends run reliably without missing GPU kernels
    device = torch.device("cpu")
    print(f"Running 4-Way Forward & Backward Benchmark on: {device.type.upper()}\n")

    # Dimensions (Scaled up to see structural differences clearly)
    vocab_size = 1000
    embed_dim = 128
    batch_size = 64
    seq_len = 256
    iterations = 50  # Reduced slightly because Dense One-Hot is highly demanding

    print(f"Configuration -> Vocab: {vocab_size}, Embed: {embed_dim}, Batch: {batch_size}, Seq: {seq_len}")
    print("-" * 70)

    # 1. Initialize all 4 models (Assumes they exist in your current namespace)
    try:
        model_onehot = OneHotEmbedding(vocab_size, embed_dim).to(device)
        model_sparse = SparseEmbedding(vocab_size, embed_dim).to(device)
        model_gather = GatherEmbedding(vocab_size, embed_dim).to(device)
        model_native = NativeEmbedding(vocab_size, embed_dim).to(device)
    except NameError as e:
        print(f"Initialization Error: Ensure all 4 classes are defined before running. Detail: {e}")
        return

    # 2. Synchronize all weights perfectly to ensure identical math
    shared_weights = model_native.W.data.clone()
    model_onehot.W.data = shared_weights.clone()
    model_sparse.W.data = shared_weights.clone()
    model_gather.W.data = shared_weights.clone()

    # 3. Generate baseline tokens
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

    # Dictionary to keep track of execution profiles
    profiles = {
        "One-Hot Dense": {"model": model_onehot, "fw": 0.0, "bw": 0.0},
        "Sparse Math":   {"model": model_sparse, "fw": 0.0, "bw": 0.0},
        "Manual Gather": {"model": model_gather, "fw": 0.0, "bw": 0.0},
        "Native Index":  {"model": model_native, "fw": 0.0, "bw": 0.0}
    }

    # 4. Execute Benchmark Loop
    for name, profile in profiles.items():
        model = profile["model"]
        
        # --- FORWARD PASS TIME ---
        start_fw = time.perf_counter()
        for _ in range(iterations):
            # We track gradients here because we want an accurate forward profile 
            # that prepares the autograd graph for the backward pass
            out = model(x)
        profile["fw"] = (time.perf_counter() - start_fw) / iterations

        # --- BACKWARD PASS TIME ---
        start_bw = time.perf_counter()
        for _ in range(iterations):
            model.W.grad = None  # Reset tracking
            out = model(x)
            loss = out.sum()
            loss.backward()
        profile["bw"] = (time.perf_counter() - start_bw) / iterations
        
        print(f"Finished profiling: {name}")

    # 5. Display Performance Leaderboard
    print("\n" + "="*70)
    print(f"{'EMBEDDING PARADIGM':<20} | {'FORWARD AVG (s)':<18} | {'BACKWARD AVG (s)':<18}")
    print("="*70)
    for name, timings in profiles.items():
        print(f"{name:<20} | {timings['fw']:.6f}s            | {timings['bw']:.6f}s")
    print("="*70)

if __name__ == "__main__":
    run_four_way_benchmark()

Running 4-Way Forward & Backward Benchmark on: CPU

Configuration -> Vocab: 1000, Embed: 128, Batch: 64, Seq: 256
----------------------------------------------------------------------
Finished profiling: One-Hot Dense
Finished profiling: Sparse Math
Finished profiling: Manual Gather
Finished profiling: Native Index

EMBEDDING PARADIGM   | FORWARD AVG (s)    | BACKWARD AVG (s)  
One-Hot Dense        | 0.006788s            | 0.009073s
Sparse Math          | 0.000524s            | 0.001265s
Manual Gather        | 0.000500s            | 0.001069s
Native Index         | 0.000198s            | 0.001480s


> ✍️ **My Work**

### 7. Multi-Layer Perceptron (MLP)
The feed-forward network used inside each Transformer block, consisting of multiple linear layers with ReLU activations.

In [8]:
from numpy import True_
class MLP(nn.Module): 
  def __init__(self, d_model=512):
        super().__init__()
        self.L1 = Linear(d_in =d_model, d_out = d_model*2)
        self.L2 = Linear(d_in = d_model*2, d_out = d_model*2)
        self.L3 = Linear(d_in = d_model*2, d_out = d_model*4)
        self.L4 = Linear(d_in = d_model*4, d_out = d_model*2)
        self.L5 = Linear(d_in = d_model*2, d_out = d_model, last = True)
        self.layers = nn.ModuleList([
          self.L1,
          self.L2,
          self.L3,
          self.L4,
          self.L5,
          ])
  def forward(self, x): 
    for l in self.layers: 
      x = l(x)
    return x
    



> 🤖 **AI-Assisted**

### 8. Positional Encoding
Since Transformers process all tokens in parallel, they lack an inherent sense of sequence order. This injects sine and cosine frequencies so the model knows where each token is positioned.

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # Create a matrix of shape (max_len, d_model) filled with zeros
        pe = torch.zeros(max_len, d_model)
        
        # Create a vector of positions (0, 1, 2, ...) and reshape to (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Calculate the division term (denominator) for the sine and cosine formulas
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices in the array
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices in the array
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as buffer so it's saved with the model state but not treated as a trainable parameter
        # Cast the float32 math into bfloat16 before saving it!
        self.register_buffer('pe', pe.unsqueeze(0).to(torch.bfloat16))


    def forward(self, x):
        # x shape expected: (batch_size, seq_len, d_model)
        # Add the positional encoding to the input embeddings
        x = x + self.pe[:, :x.size(1)]
        return x


> ✍️ **My Work**

### 9. Custom Layer Normalization
Implementing LayerNorm from scratch by computing the mean and variance across the embedding dimension, then shifting and scaling with learnable parameters `gamma` and `beta`.

In [10]:
class LayerNorm(nn.Module):
    def __init__(self,  d_model=512, eps = 1e-5):
        super().__init__()
        self.beta = nn.Parameter(
                                torch.zeros(
                                        size = (d_model, ),
                                        requires_grad=True, 
                                        dtype = torch.bfloat16
                                ) )
        self.gamma = nn.Parameter(
                                torch.ones(
                                        size = (d_model, ),
                                        requires_grad=True, 
                                        dtype = torch.bfloat16
                                ) )
        self.eps = eps
    def forward(self, x): 
        # over non batch
        meu =   torch.mean( x, dim = -1, keepdim = True)
        sigma = torch.var(x, dim = -1, keepdim = True)
        x_hat = (x - meu ) / torch.sqrt(sigma + self.eps)
        return self.gamma * x_hat + self.beta



> ✍️ **My Work**

### 10. Multi-Head Causal Attention
The core of the Transformer! This computes scaled dot-product attention. Notice the `causal_mask` which prevents tokens from 'looking ahead' into the future during autoregressive training.

In [11]:
class MultiHeadAttention(nn.Module): 
    def __init__(self,  d_model=512, n = 8, mask = True):
        super().__init__()
        assert d_model%n ==0 
        self.n = n
        self.d_model = d_model
        self.mask = mask
        self.K = Linear(d_model, d_model, last = True)
        self.Q = Linear(d_model, d_model, last = True)
        self.V = Linear(d_model, d_model, last = True)
        self.O = Linear(d_model, d_model, last = True)

        self.h_dim = d_model//n 
    def forward(self, x):
                B, seq_len, _= x.shape
                K= self.K(x)
                V= self.V(x)
                Q= self.Q(x)
                # x:(B , seq_len, d_model) 
                # q: (B , seq_len, d_model, d_model) - > (B, seq_len, n, h_dim ) 
                K = K.view(B, seq_len, self.n, self.d_model//self.n)
                V = V.view(B, seq_len, self.n, self.d_model//self.n)
                Q = Q.view(B, seq_len, self.n, self.d_model//self.n)
                # q: (B, seq_len, n, h_dim ) - > (B, n, seq_len h_dim) 
                K = K.transpose(1,2)
                V = V.transpose(1,2)
                Q = Q.transpose(1,2)
                # q: (B, n, seq_len, h_dim) x k: (B, n,  h_dim, seq_len)  - > (B, n, seq_len, seq_len)

                scores = Q.matmul(K.transpose(-2, -1)) / math.sqrt(self.d_model// self.n)
                # q: (B, n, seq_len, h_dim) x k: (B, n,  h_dim, seq_len)  - > (B, n, seq_len, seq_len)

                if self.mask:
                    causal_mask = torch.tril(torch.ones(size = (seq_len, seq_len), device = x.device))
                    scores = scores.masked_fill( causal_mask==0, float('-inf') )


                # softmax over position 3 : seq_len
                sm = F.softmax(scores , dim =-1)
                # softmax: (B, n, seq_len, seq_len) x (B, n, seq_len, h_dim) -> (B, n, seq_len, h_dim) 
                attn = sm.matmul(V) 
                # communication is over now we bring it back to orignal x shape 
                #(B, n, seq_len, h_dim) -> (B, seq_len, d_model) 
                # (B, n, seq_len, h_dim) -> (B, seq_len, n,  h_dim) 
                x = attn.transpose(1,2)
                # 0    1     2      3 
                #(B, seq_len, n,  h_dim)  -> (B, seq_len, d_model) 
                x = x.contiguous().flatten(start_dim = 2, end_dim= 3)

                return self.O(x)

> ✍️ **My Work**

### 11. Full Transformer Assembly
Bringing it all together: Embedding + Positional Encoding -> LayerNorm -> Masked Attention -> LayerNorm -> MLP -> Output Projection.

In [12]:
class GPTTransformer(nn.Module): 
        """ 
        """
        def __init__(self, e = 1024 , d_model=512, n = 8, v =65, cw = 2048):
                super().__init__()
                self.pos_enc= PositionalEncoding(max_len=cw, d_model=d_model)
                self.emb = Embedding(d_in = v, d_out= d_model)
                self.masked_attn = MultiHeadAttention(d_model= d_model, n = n)
                self.ln1 = LayerNorm(d_model)
                self.mlp = MLP(d_model)
                self.ln2 = LayerNorm(d_model)     
                self.linear = Linear(d_model, v)

        def forward(self, x): 
                # embed
                print(x)
                x = self.emb(x)
                # pos enc
                x = x + self.pos_enc(x)
                # masked attn
                x = x + self.masked_attn(self.ln1(x))
                # mlp 
                x = x + self.mlp(self.ln2(x))
                # project to vocab
                x = self.linear(x)
                # return raw logits
                return  x


> ✍️ **My Work**

### 12. Model Instantiation
Creating the model and moving it to the target device.

In [13]:
model = GPTTransformer()
model.to(device)

GPTTransformer(
  (pos_enc): PositionalEncoding()
  (emb): Embedding()
  (masked_attn): MultiHeadAttention(
    (K): Linear()
    (Q): Linear()
    (V): Linear()
    (O): Linear()
  )
  (ln1): LayerNorm()
  (mlp): MLP(
    (L1): Linear()
    (L2): Linear()
    (L3): Linear()
    (L4): Linear()
    (L5): Linear()
    (layers): ModuleList(
      (0-4): 5 x Linear()
    )
  )
  (ln2): LayerNorm()
  (linear): Linear()
)

> ✍️ **My Work**

### 13. Optimizer Setup
Using AdamW for gradient descent optimization.

In [14]:
optimizer = torch.optim.AdamW(lr = 0.001, params= model.parameters())

> ✍️ **My Work**

### 14. Single GPU Training Loop
A standard training loop computing cross-entropy loss, backpropagating gradients, and updating weights. This serves as our baseline benchmark.

In [15]:
%%time

EPOCHS = 1

for i in range(EPOCHS):
    total_loss = 0
    model.train()

    for batch in dataloader:
        batch_x = batch[0].to(device)
        batch_y = batch[1].to(device)
        
        y_pred = model(batch_x) 
        
        loss = F.cross_entropy(y_pred.view(-1, vocab_size), batch_y.view(-1))   

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()
        
        # Add to total loss (.item() pulls the float out of the PyTorch tensor)
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    
    print(f"Epoch {i+1} | Average Loss: {avg_loss:.4f}")
    print(f"Samples covered: {len(dataloader)}")



tensor([[44, 43, 41,  ..., 17, 26, 21],
        [10,  0, 15,  ...,  1, 58, 46],
        [43, 50, 50,  ..., 57,  1, 39],
        ...,
        [42, 47, 42,  ..., 47, 58, 47],
        [58, 57,  1,  ...,  0, 26, 53],
        [43, 52, 12,  ..., 57, 46,  1]], device='mps:0')
tensor([[53, 59, 50,  ...,  1, 46, 43],
        [ 0, 63, 53,  ...,  1, 40, 43],
        [56,  6,  0,  ..., 52, 42,  1],
        ...,
        [ 1, 47, 57,  ..., 59,  1, 41],
        [56, 53, 53,  ..., 53, 52,  8],
        [35, 46, 53,  ..., 43, 41, 46]], device='mps:0')
tensor([[ 1, 58, 53,  ..., 59, 58,  6],
        [ 1, 58, 46,  ..., 32, 56, 59],
        [43, 57, 11,  ..., 43, 56,  1],
        ...,
        [56, 43,  1,  ..., 42,  1, 47],
        [ 1, 43, 52,  ..., 63,  6,  7],
        [43, 50, 50,  ..., 40, 50, 47]], device='mps:0')
tensor([[40, 57,  0,  ..., 59, 54, 54],
        [39, 58,  1,  ...,  6,  1, 47],
        [57, 58,  1,  ...,  0, 25, 17],
        ...,
        [39, 44, 44,  ..., 17, 26, 17],
        [ 1, 61, 

KeyboardInterrupt: 

> ✍️ **My Work**

### 15. DDP Initialization
Setting up the distributed process group using the `nccl` backend. This is required before any distributed communication can happen.

In [ ]:
def ddp_setup(rank , world_size ): 
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12323"
    init_process_group(backend="nccl", rank = rank, world_size = world_size)



 

> ✍️ **My Work**

### 13. Optimizer Setup
Using AdamW for gradient descent optimization.

In [ ]:
def run ( rank, world_size):
    ddp_setup(rank, world_size)
    EPOCHS = 1
    dataset = CharDataset(data, seq_len)
    sampler = DistributedSampler(dataset)
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, sampler =sampler)

    model = DDP(GPTTransformer().to(rank) , device_ids = [rank])

    optimizer = torch.optim.AdamW(lr = 0.001, params= model.parameters())

    for i in range(EPOCHS):
        total_loss = 0
        model.train()
        sampler.set_epoch(i)
        for batch in dataloader:
            batch_x = batch[0].to(rank)
            batch_y = batch[1].to(rank)
            
            y_pred = model(batch_x) 
            
            loss = F.cross_entropy(y_pred.view(-1, vocab_size), batch_y.view(-1))   

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()
            
            # Add to total loss (.item() pulls the float out of the PyTorch tensor)
            total_loss += loss.item()
            
        avg_loss = total_loss / len(dataloader)
        if rank == 0:
            print(f"Epoch {i+1} | Average Loss: {avg_loss:.4f}")
            print(f"Samples/GPU: {len(dataloader)}, Total: {len(dataloader)*world_size}")


    destroy_process_group()

> ✍️ **My Work**

### 17. Multi-Processing Execution
Spawning the DDP processes across all available GPUs. We synchronize before and after to get an accurate measurement of the wall-clock time.

In [ ]:
%%time
torch.cuda.synchronize()
start = time.time()
world_size = torch.cuda.device_count()
mp.start_processes(run, args=(world_size,), nprocs=world_size, start_method='spawn')
torch.cuda.synchronize()
end = time.time()


In [ ]:
print(end-start)